In [1]:
import polars as pl
import pandas as pd

In [2]:
df_erp = pl.read_excel("./data/Fichier_erp.xlsx")
df_web = pl.read_excel("./data/Fichier_web.xlsx")
df_liaison = pl.read_excel("./data/fichier_liaison.xlsx")

Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 21, falling back to string


In [3]:
# --- Fonction de profiling ---

def renseigne(df): 
    n = df.height
    nulls = df.null_count().row(0)
    profil = pl.DataFrame({
        'colonne': df.columns, 
        'type':[str(t)for t in df.dtypes],
        'longueur_max':[df[c].cast(pl.Utf8).str.len_chars().max() for c in df.columns],
        'non_null':[n - x for x in nulls],
        'nan':list(nulls),
        "% nan": [round(x / n * 100, 2) for x in nulls],
        'valeur_unique':[df[c].n_unique()for c in df.columns]
    })
    return profil

In [4]:
renseigne(df_erp)


colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""product_id""","""Int64""",4,825,0,0.0,825
"""onsale_web""","""Int64""",1,825,0,0.0,2
"""price""","""Float64""",5,825,0,0.0,382
"""stock_quantity""","""Int64""",3,825,0,0.0,129
"""stock_status""","""String""",10,825,0,0.0,2


In [5]:
renseigne(df_web)


colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""sku""","""Int64""",5,1424,89,5.88,713
"""virtual""","""Int64""",1,1513,0,0.0,1
"""downloadable""","""Int64""",1,1513,0,0.0,1
"""rating_count""","""Int64""",1,1513,0,0.0,1
"""average_rating""","""Int64""",1,1430,83,5.49,2
…,…,…,…,…,…,…
"""guid""","""String""",133,1430,83,5.49,1430
"""menu_order""","""Int64""",1,1430,83,5.49,2
"""post_type""","""String""",10,1430,83,5.49,3


In [6]:
renseigne(df_liaison)

colonne,type,longueur_max,non_null,nan,% nan,valeur_unique
str,str,i64,i64,i64,f64,i64
"""product_id""","""Int64""",4,825,0,0.0,825
"""id_web""","""Int64""",5,731,94,11.39,732


In [7]:
tables = {
    "erp": df_erp,
    "web": df_web,
    "liaison": df_liaison,
}

with pd.ExcelWriter("./data/profiling_p10_raw.xlsx") as writer:
    for nom, df in tables.items():
        profil = renseigne(df)
        profil.to_pandas().to_excel(writer, sheet_name=nom[:31], index=False)
